# FlashAttention: IO-Aware Attention - 실습 코드 1: FlashAttention 사용법 (PyTorch)

- Tutorial ID: `expand-flash-attention`
- Tutorial: FlashAttention: IO-Aware Attention
- Section ID: `expand-flash-attention-code-1`
- Section: 실습 코드 1: FlashAttention 사용법 (PyTorch)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: FlashAttention 사용법 (PyTorch)
#
# 이 노트북은 "정답 코드를 한 번 실행"하는 용도가 아니라,
# "왜 FlashAttention이 필요한가" 부터 "실제로 어떻게 사용하는가"까지
# 개념 -> 코드 -> 검증의 순서로 한 걸음씩 따라가는 실습 노트입니다.
# (이전 버전보다 설명과 예제가 훨씬 자세해졌습니다.)
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 직접 추적한다
#   2) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 정확히 0이 되는지 눈으로 확인한다
#   3) attention 연산이 왜 메모리를 많이 쓰는지, GPU 메모리 구조(HBM/SRAM)로 설명한다
#   4) PyTorch에서 FlashAttention을 직접 호출하고, naive 구현과 결과/메모리/속도를 비교한다
#
# 노트북 구성 (아래 순서대로 읽으면 됩니다):
#   STEP 0.  환경 설정 (device, seed)
#   STEP 1.  Scaled Dot-Product Attention 복습 (수식과 의미)
#   STEP 2.  Naive Attention을 직접 구현하며 shape/값 추적
#   STEP 3.  왜 문제가 되는가? — attention 행렬의 메모리 계산
#   STEP 4.  IO-Awareness — FlashAttention의 핵심 아이디어
#   STEP 5.  PyTorch에서 FlashAttention 사용하기 (SDPA)
#   STEP 6.  naive 구현과 결과값이 정말 같은지 검증
#   STEP 7.  메모리 사용량 비교
#   STEP 8.  속도 비교
#   STEP 9.  직접 실험해보기 (seq_len을 바꿔가며 관찰)
#   STEP 10. 핵심만 다시 보기 (Cheat Sheet)
#   STEP 11. 정리 및 참고문헌
#
# 주의:
#   - 숫자 하나하나를 외우기보다 "shape 변화"와 "정보가 이동하는 방향"을 보세요.
#   - FlashAttention은 원래 CUDA GPU의 메모리 구조를 활용하도록 설계된 최적화입니다.
#     GPU가 없어도 대부분의 코드는 실행되지만(자동으로 다른 구현으로 대체됨),
#     이 노트북이 보여주려는 '진짜' 메모리/속도 절약 효과는
#     CUDA GPU 환경(Google Colab의 GPU 런타임 등)에서 가장 뚜렷하게 나타납니다.
#     STEP 0에서 현재 환경을 자동으로 확인하고 알려드립니다.


In [ ]:
# ============================================================
# STEP 0. 환경 설정 (Setup)
# ============================================================
# 이 노트북에서 사용할 라이브러리를 불러오고, 실행 환경(GPU 유무)을 확인합니다.

import torch
import torch.nn.functional as F
import math
import time

# ------------------------------------------------------------
# [중요] FlashAttention은 원래 "CUDA GPU 전용" 커널로 설계되었습니다.
#
# FlashAttention은 NVIDIA GPU의 메모리 구조(HBM ↔ SRAM, STEP 4에서 자세히 설명)를
# 직접 활용하도록 만들어진 저수준(low-level) 커널입니다. 그래서 CUDA GPU가 없는
# 환경에서는 PyTorch가 자동으로 다른 구현(주로 "MATH" backend, 즉 우리가
# STEP 2에서 직접 만들어볼 naive 구현과 비슷한 방식)으로 대체해서 실행합니다.
# (PyTorch 버전에 따라 CPU용 FLASH_ATTENTION 구현이 존재하기도 하지만,
#  이 노트북이 설명하는 GPU HBM/SRAM 기반의 절약 효과와는 다른 이야기입니다.
#  STEP 5-2에서 현재 환경이 실제로 무엇을 지원하는지 직접 확인해봅니다.)
#
# 즉, 이 노트북의 "메모리/속도 비교" 결과를 제대로(극적으로) 관찰하려면
# CUDA GPU 환경(Google Colab의 GPU 런타임 등)이 가장 좋습니다.
# GPU가 없어도 코드 대부분은 정상적으로 실행되지만, 몇몇 비교 실험은
# 안내 메시지로 대체됩니다.
# ------------------------------------------------------------

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"[INFO] CUDA 사용 가능 (device: {torch.cuda.get_device_name(0)})")
    print("[INFO] 이 노트북의 모든 비교 실험이 의미 있게 동작합니다.")
else:
    device = torch.device("cpu")
    print("[WARNING] CUDA를 사용할 수 없어 CPU로 실행합니다.")
    print("[WARNING] 위 설명대로, 이 노트북의 메모리/속도 비교 실험 중 일부는")
    print("          CUDA 환경 전용이라 안내 메시지로 대체됩니다.")
    print("          -> Google Colab에서 [런타임 > 런타임 유형 변경 > GPU]를 선택하는 것을 권장합니다.")

# 실습 결과를 항상 동일하게 재현하기 위해 random seed를 고정합니다.
# (seed를 고정하지 않으면 매번 실행할 때마다 난수가 달라져서
#  같은 코드를 실행해도 값이 조금씩 달라 보일 수 있습니다.)
SEED = 42
torch.manual_seed(SEED)
print(f"[INFO] Random seed = {SEED} 로 고정했습니다.")


## STEP 1. Scaled Dot-Product Attention 복습

FlashAttention은 "새로운 수학 공식"이 아니라, 기존의 attention 연산을
**GPU 메모리를 효율적으로 쓰도록 다시 구현한 것**입니다. 그래서 먼저
우리가 최적화하려는 대상, 즉 "Scaled Dot-Product Attention"이 정확히
어떤 계산인지 다시 짚고 넘어갑니다.

### 기본 아이디어

Attention은 "지금 이 토큰이 다른 토큰들을 얼마나 참고해야 하는가"를
계산하는 연산입니다. 이를 위해 3가지 벡터를 사용합니다.

- **Q (Query)**: "내가 지금 찾고 있는 정보는 무엇인가?" — 질문 역할
- **K (Key)**: "각 토큰이 가진 정보의 색인(index)표" — 검색 대상 역할
- **V (Value)**: "각 토큰이 실제로 담고 있는 내용물" — 진짜로 가져올 정보

도서관에서 책을 찾는 상황에 비유할 수 있습니다.

- Q = 내가 찾고 싶은 책의 "주제" (질문)
- K = 서가에 붙어 있는 "책 제목표" (색인)
- V = 그 서가에 실제로 꽂혀 있는 "책 내용"

내가 가진 질문(Q)을 각 서가의 색인표(K)와 비교해서 "얼마나 관련있는지"
점수를 매기고(=유사도), 그 점수를 확률로 바꿔 V들을 가중합(weighted sum)
하면 "지금 질문에 대한 답"을 얻을 수 있습니다.

### 수식

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

각 기호의 의미는 다음과 같습니다.

- $Q \in \mathbb{R}^{N \times d_k}$ : Query 행렬 ($N$ = 시퀀스 길이, 즉 토큰 개수)
- $K \in \mathbb{R}^{N \times d_k}$ : Key 행렬
- $V \in \mathbb{R}^{N \times d_v}$ : Value 행렬
- $QK^\top \in \mathbb{R}^{N \times N}$ : 모든 토큰 쌍(pair)의 유사도 점수
  = **attention score**
- $\sqrt{d_k}$ 로 나누는 이유: 벡터 차원 $d_k$가 커질수록 내적(dot product)
  값도 커지는 경향이 있어, 이를 적절히 눌러 softmax가 한쪽으로
  치우치는 것을 막기 위한 스케일 조정입니다.
- $\text{softmax}$ : 점수를 "합이 1인 확률"로 변환
- 마지막에 $V$를 곱해서, 확률(가중치)만큼 각 토큰의 V를 섞은 결과를 얻습니다.

바로 이 계산 과정에서 나오는 "$N \times N$ 크기의 attention score 행렬"이
**FlashAttention이 해결하려는 핵심 문제**입니다. 왜 문제가 되는지
STEP 2에서 직접 만들어보면서 눈으로 확인해봅시다.


In [ ]:
# ============================================================
# STEP 2. Naive Attention을 직접 구현하며 shape 추적하기
# ============================================================
# 아주 작은 예시로 시작합니다. 실제 모델은 수백~수천 개의 토큰을 다루지만,
# 여기서는 "무슨 일이 벌어지는지" 눈으로 직접 확인하기 위해
# 일부러 작은 숫자를 사용합니다.
#
# (아래에서는 뒤에 나올 실제-크기 예제와 구분하기 위해 대문자 변수명을 사용합니다.
#  STEP 5부터는 실제 모델에 가까운 크기를 다루면서 소문자 q, k, v를 사용할 예정입니다.)

batch_size = 1   # 문장 1개만 처리
num_heads = 1    # attention head 1개만 사용 (multi-head는 이 계산을 여러 벌 병렬로 반복하는 것뿐입니다)
seq_len = 6      # 토큰 6개짜리 아주 짧은 문장이라고 가정
d_k = 8          # 각 토큰을 8차원 벡터로 표현

# Q, K, V를 무작위 값으로 생성합니다.
# 실제로는 "입력 문장의 embedding" @ "학습된 가중치 행렬(W_Q, W_K, W_V)"로
# Q, K, V가 만들어지지만, 여기서는 "이미 만들어진 결과"라고 가정하고
# 랜덤 텐서로 대체해 attention 연산 자체에 집중합니다.
Q = torch.randn(batch_size, num_heads, seq_len, d_k, device=device)
K = torch.randn(batch_size, num_heads, seq_len, d_k, device=device)
V = torch.randn(batch_size, num_heads, seq_len, d_k, device=device)

print("=== 입력 shape 확인 ===")
print(f"Q shape: {tuple(Q.shape)}  # (batch_size, num_heads, seq_len, d_k)")
print(f"K shape: {tuple(K.shape)}")
print(f"V shape: {tuple(V.shape)}")
print()
print("Q의 실제 값 (batch=0, head=0):")
print(Q[0, 0])


In [ ]:
# ------------------------------------------------------------
# 2-1) Q @ K^T : 모든 토큰 쌍(pair)의 유사도(attention score) 계산
# ------------------------------------------------------------
# K.transpose(-2, -1)은 K의 마지막 두 차원(seq_len, d_k)의 순서를 뒤바꿔
# (batch, heads, d_k, seq_len) 형태로 만듭니다.
# 그래야 (..., seq_len, d_k) @ (..., d_k, seq_len) = (..., seq_len, seq_len)
# 형태의 행렬곱이 가능해집니다. (행렬곱은 안쪽 차원끼리 맞아야 합니다)
scores = Q @ K.transpose(-2, -1)

print("=== Q @ K^T 결과 ===")
print(f"scores shape: {tuple(scores.shape)}  # (batch, heads, seq_len, seq_len)")
print("-> 중요한 포인트: seq_len이 6인데, 결과는 6 x 6 행렬이 되었습니다.")
print("   즉, 6개 토큰이 서로에게 '모든 쌍(pair)'에 대해 점수를 매긴 것입니다.")
print("   토큰이 N개면 이 행렬의 크기는 N x N 이 됩니다. (STEP 3에서 이게 왜 문제인지 다룹니다)")

# ------------------------------------------------------------
# 2-2) 스케일링 (Scaling): sqrt(d_k)로 나누기
# ------------------------------------------------------------
# d_k(벡터 차원)가 커질수록 내적(dot product) 값의 분산도 커져서,
# 다음 단계인 softmax의 입력값이 지나치게 커지는 경향이 있습니다.
# softmax는 입력값의 차이가 크면 확률이 한쪽으로 쏠려버리기 때문에,
# sqrt(d_k)로 나눠 값의 크기를 적절히 눌러줍니다.
scaled_scores = scores / math.sqrt(d_k)

print()
print("=== 스케일링 후 ===")
print(f"scaled_scores shape: {tuple(scaled_scores.shape)}  (shape은 그대로, 값의 크기만 작아짐)")
print(f"스케일링 전 평균 절댓값: {scores.abs().mean().item():.4f}")
print(f"스케일링 후 평균 절댓값: {scaled_scores.abs().mean().item():.4f}")


In [ ]:
# ------------------------------------------------------------
# 2-3) Causal Mask 적용: "미래 토큰을 보지 못하게 막기"
# ------------------------------------------------------------
# GPT 같은 언어 모델은 왼쪽에서 오른쪽으로 문장을 한 토큰씩 생성합니다.
# 즉, 3번째 토큰을 예측할 때 4, 5, 6번째 토큰(미래)을 미리 봐서는 반칙입니다.
# (참고: BERT처럼 문장을 양방향으로 한 번에 보는 encoder 모델은 이 마스크를 쓰지 않습니다.)
#
# 이를 막기 위해 "미래에 해당하는 위치"의 점수를 -inf(음의 무한대)로 바꿔줍니다.
# -inf는 잠시 후 softmax를 통과하면 정확히 0이 되기 때문입니다. (STEP 2-4에서 확인)

# torch.triu: upper triangular, 즉 "대각선 기준 오른쪽 위 부분만 True로 남기는" 함수
# diagonal=1 옵션은 "대각선 바로 위 칸부터" 마스킹한다는 뜻입니다.
# (자기 자신, 즉 대각선 위치는 봐도 되므로 대각선 자체는 막지 않습니다)
causal_mask = torch.triu(
    torch.ones(seq_len, seq_len, device=device, dtype=torch.bool),
    diagonal=1
)

print("=== Causal mask (True = 가려지는 위치, 즉 볼 수 없는 미래) ===")
print(causal_mask)
print()
print("해석: 행(row) = 지금 보고 있는 토큰, 열(column) = 참고하려는 토큰이라고 하면,")
print("      1행(첫 번째 토큰)은 자기 자신(1번)만 볼 수 있고 2~6번(미래)은 모두 가려짐(True)")
print("      3행(세 번째 토큰)은 1,2,3번은 볼 수 있고 4,5,6번(미래)은 가려짐")
print("      -> 대각선 위쪽만 True(가려짐)인 계단 모양이 보이시나요?")

# masked_fill: mask가 True인 위치를 지정한 값(-inf)으로 채워 넣는 함수
masked_scores = scaled_scores.masked_fill(causal_mask, float('-inf'))

print()
print("=== Mask 적용 후 scores (batch=0, head=0) ===")
print(masked_scores[0, 0])
print("-> 대각선 위쪽(미래 위치)이 모두 -inf로 바뀐 것을 확인하세요.")


In [ ]:
# ------------------------------------------------------------
# 2-4) Softmax: 점수를 확률로 변환
# ------------------------------------------------------------
# dim=-1은 "마지막 차원(=한 행에 있는 6개 값)을 기준으로 softmax를 적용"한다는 뜻입니다.
# 즉, 토큰(행)마다 "내가 참고할 수 있는 토큰들에 대한 확률 분포"를 하나씩 만듭니다.
attn_weights = F.softmax(masked_scores, dim=-1)

print("=== Softmax 결과 (batch=0, head=0) ===")
print(attn_weights[0, 0])
print()
print("확인 포인트 1: -inf였던 위치들이 정확히 0.0이 되었습니다!")
print("               (수학적으로 e^(-inf) = 0 이기 때문입니다)")
print("확인 포인트 2: 각 행의 합이 1.0인지 확인해봅니다 (softmax의 정의: 확률의 합은 1).")
print(f"               각 행의 합: {attn_weights[0, 0].sum(dim=-1)}")

# assert로 "미래 토큰의 확률이 정말 0인지"를 코드로도 자동 검증합니다.
assert torch.all(attn_weights[0, 0][causal_mask] == 0.0), "미래 토큰의 확률이 0이 아닙니다!"
print()
print("[검증 성공] 미래 토큰에 대한 attention 확률이 모두 0임을 확인했습니다.")
print("            -> 이 모델은 절대 '미래를 커닝'할 수 없다는 뜻입니다.")


In [ ]:
# ------------------------------------------------------------
# 2-5) 확률(attn_weights)과 V를 곱해 최종 출력 계산
# ------------------------------------------------------------
naive_output = attn_weights @ V

print("=== 최종 출력 ===")
print(f"attn_weights shape: {tuple(attn_weights.shape)}  # (batch, heads, seq_len, seq_len)")
print(f"V shape:            {tuple(V.shape)}             # (batch, heads, seq_len, d_k)")
print(f"output shape:       {tuple(naive_output.shape)}  # (batch, heads, seq_len, d_k)")
print()
print("-> 입력 Q와 똑같은 shape (batch, heads, seq_len, d_k)의 결과가 나왔습니다.")
print("   각 토큰이 '자신이 볼 수 있는 토큰들의 V를 확률적으로 섞은 값'으로 바뀐 것입니다.")

# ------------------------------------------------------------
# 지금까지의 5단계를 재사용하기 쉽도록 하나의 함수로 정리합니다.
# (뒤에서 FlashAttention과 결과를 비교할 때 이 함수를 다시 사용합니다)
# ------------------------------------------------------------
def naive_scaled_dot_product_attention(Q, K, V, is_causal=False):
    """
    표준(naive) 방식의 Scaled Dot-Product Attention.

    핵심 특징: (seq_len, seq_len) 크기의 attention score 행렬을
    "통째로" 메모리에 만든 뒤 계산합니다.
    -> 이것이 FlashAttention과의 가장 큰 차이점입니다. (STEP 4에서 자세히 다룹니다)
    """
    d_k = Q.shape[-1]
    seq_len = Q.shape[-2]

    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

    if is_causal:
        mask = torch.triu(
            torch.ones(seq_len, seq_len, device=Q.device, dtype=torch.bool),
            diagonal=1
        )
        scores = scores.masked_fill(mask, float('-inf'))

    attn_weights = F.softmax(scores, dim=-1)
    output = attn_weights @ V
    return output


# 함수가 위에서 직접 계산한 것과 같은 결과를 내는지 확인합니다.
output_from_function = naive_scaled_dot_product_attention(Q, K, V, is_causal=True)
print()
print("함수 결과와 수동 계산 결과가 같은가?",
      torch.allclose(output_from_function, naive_output))


## STEP 3. 왜 이것이 문제가 될까요?

STEP 2에서 확인한 것처럼, attention을 계산하려면
**(seq_len × seq_len) 크기의 행렬**을 반드시 한 번은 만들어야 하는 것처럼 보입니다.
(scores, masked_scores, attn_weights 모두 이 크기였습니다.)

문제는, 실제 LLM에서는 seq_len이 6이 아니라 수천에서 많게는 수십만이라는 점입니다.
seq_len이 커질수록 이 행렬은 **제곱(quadratic)** 으로 커집니다.
아래 코드로 실제 크기를 계산해봅시다.


In [ ]:
# ============================================================
# STEP 3. Attention 행렬의 메모리 사용량 계산
# ============================================================
# attention score 행렬 1개의 크기 = batch_size * num_heads * seq_len * seq_len
# float16(원소 1개당 2 byte) 기준으로 실제 메모리(MB)를 계산해봅니다.

def attention_matrix_memory_mb(batch_size, num_heads, seq_len, bytes_per_element=2):
    """
    (batch, heads, seq_len, seq_len) 크기 attention 행렬 1개가
    차지하는 메모리를 MB 단위로 계산합니다.
    bytes_per_element=2 는 float16 기준입니다. (float32라면 4를 사용)
    """
    num_elements = batch_size * num_heads * seq_len * seq_len
    total_bytes = num_elements * bytes_per_element
    return total_bytes / (1024 ** 2)


print(f"{'seq_len':>8} | {'attention 행렬 메모리 (batch=2, heads=8 기준)':>40}")
print("-" * 55)
for seq_len_example in [128, 512, 1024, 2048, 4096, 8192, 16384]:
    mem_mb = attention_matrix_memory_mb(batch_size=2, num_heads=8, seq_len=seq_len_example)
    print(f"{seq_len_example:>8} | {mem_mb:>37.1f} MB")

print()
print("-> seq_len이 2배가 될 때마다, 메모리는 약 '4배'로 늘어납니다. (N^2 이기 때문)")
print("   게다가 STEP 2에서 직접 본 것처럼, scores / masked_scores / attn_weights 등")
print("   이 크기의 행렬이 계산 과정에서 여러 개 만들어질 수 있어 실제 사용량은 더 큽니다.")
print("   또한 학습(backward) 시에는 이 값들 중 일부를 저장해둬야 할 수도 있어")
print("   메모리 문제는 더 심각해집니다.")


## STEP 4. FlashAttention의 핵심 아이디어: "IO-Awareness"

지금까지 확인한 문제를 정리하면:
> attention을 계산하려면 (seq_len × seq_len) 크기의 큰 행렬을
> 메모리에 만들어야 하고, 이는 시퀀스가 길어질수록 급격히(N² 비율로) 커진다.

FlashAttention은 "이 계산 자체를 생략하는 방법"을 찾은 것이 **아닙니다.**
수학적으로는 STEP 2에서 만든 naive 구현과 **완전히 동일한 결과**를 냅니다
(그래서 논문 제목에도 "Exact Attention"이라는 표현이 들어갑니다 — 근사치가
아니라 정확히 같은 값이라는 뜻입니다).

대신 FlashAttention은 **"그 큰 행렬을 GPU의 느린 메모리에 통째로
쓰고 읽지 않으면서 계산하는 방법"**을 찾았습니다. 이것이 논문 제목의
"IO-Aware"(입출력을 인식하는)라는 표현의 의미입니다.

### GPU 메모리는 한 종류가 아닙니다

GPU 안에는 속도와 용량이 다른 여러 종류의 메모리가 있습니다. FlashAttention
논문(Dao et al., 2022)은 NVIDIA A100 GPU를 예로 들어 이렇게 설명합니다.

| 메모리 종류 | 대략적인 용량 | 대역폭(속도) | 비유 |
|---|---|---|---|
| **HBM** (High Bandwidth Memory) | 40~80GB | 1.5~2.0 TB/s | 큰 창고 |
| **SRAM** (on-chip, GPU 코어 근처) | 약 20MB | 약 19 TB/s | 작업대 위 작은 공간 |

우리가 흔히 "GPU 메모리 사용량"이라 부르는 것(`torch.cuda.max_memory_allocated()`
등으로 측정하는 값)은 대부분 **HBM**을 가리킵니다. Q, K, V,
그리고 (seq_len × seq_len) attention score 행렬은 보통 이 HBM에 저장됩니다.
반면 SRAM은 HBM보다 10배 이상 빠르지만, 용량은 HBM의 수천분의 1 수준으로 작습니다.

### Naive 구현의 진짜 병목: "계산"이 아니라 "이동"

naive 방식은 STEP 2에서 직접 확인한 순서 그대로 다음과 같이 동작합니다.

1. HBM에서 Q, K를 읽어와 `scores = Q @ K^T` 계산 → 결과를 다시 HBM에 저장
2. HBM에서 scores를 다시 읽어와 masking → 결과를 다시 HBM에 저장
3. HBM에서 masked_scores를 다시 읽어와 softmax → 결과를 다시 HBM에 저장
4. HBM에서 attn_weights를 다시 읽어와 V와 곱함 → 최종 결과를 HBM에 저장

(seq_len × seq_len) 크기의 큰 행렬을 **HBM에 쓰고, 다시 읽고, 또 쓰고,
또 읽는** 과정이 여러 번 반복되는 것이 보이시나요? GPU의 실제 연산(곱셈,
덧셈) 속도는 매우 빠르지만, **HBM으로 데이터를 옮기는 시간(=IO)이 오히려
병목**이 됩니다. 이것이 FlashAttention 논문이 지적한 핵심 문제입니다.


### FlashAttention의 해법: Tiling(타일링) + 한 번에 처리(Fusion)

FlashAttention은 다음과 같이 동작합니다.

1. Q, K, V를 통째로 다루지 않고, **작은 블록(tile) 단위로 잘라** 처리합니다.
2. 각 블록은 크기가 작아서 **SRAM(빠른 메모리)에 통째로 올라갈 수 있습니다.**
3. `matmul → mask → softmax → matmul` 과정을 **SRAM 안에서 한 번에(fused)**
   처리하고, 중간 결과(scores, masked_scores 등)를 HBM에 저장하지 않습니다.
4. 오직 **최종 출력만** HBM에 씁니다.

이렇게 하면:
- HBM과 주고받는 데이터의 양(IO)이 크게 줄어들어 **더 빨라지고**,
- (seq_len × seq_len) 크기의 중간 행렬을 통째로 저장할 필요가 없어져
  **메모리도 절약**됩니다.

> **참고 (online softmax):** softmax는 원래 "행 전체"의 값을 한 번에 봐야
> 정확히 계산할 수 있는 연산입니다 (분모에 전체 합이 들어가기 때문입니다).
> 그런데 FlashAttention은 데이터를 블록 단위로 나누어 처리한다고 했습니다.
> "행 전체를 아직 다 보지 못했는데 softmax를 어떻게 계산하지?"라는 의문이
> 생길 수 있습니다. FlashAttention은 블록을 하나씩 볼 때마다 지금까지의
> 결과를 조금씩 "고쳐 쓰는" online softmax라는 기법으로 이 문제를 해결합니다.
> 이 노트북에서는 사용법과 효과 확인에 집중하고, online softmax의 수식은
> 다음 이론 노트북에서 직접 구현하며 자세히 다룰 예정입니다.

FlashAttention 논문은 GPT-2 모델의 attention 연산 기준으로
**7.6배의 속도 향상**을 보고했습니다. 실제 효과가 얼마나 되는지는
아래 STEP 7~9에서 여러분의 환경으로 직접 측정해봅니다.

핵심 요약: **"같은 계산을, 메모리를 아껴가며 하는 방법"** —
이것이 FlashAttention입니다.


## STEP 5. PyTorch에서 FlashAttention 사용하기

이제 개념을 알았으니 실제로 사용해봅시다.
좋은 소식은, PyTorch 2.0부터는 FlashAttention을 쓰기 위해
별도의 라이브러리를 설치할 필요 없이
`torch.nn.functional.scaled_dot_product_attention`
(줄여서 **SDPA**) 함수 하나로 사용할 수 있다는 것입니다.

### SDPA는 "하나의 함수, 여러 개의 backend"

SDPA를 호출하면 PyTorch가 내부적으로 아래 중 하나의 구현(backend)을
**자동으로 선택**합니다.

| Backend | 설명 | 동작 환경 |
|---|---|---|
| `MATH` | STEP 2에서 직접 만든 것과 비슷한, 가장 단순하고 이식성 높은 구현 | CPU/GPU 모두 |
| `FLASH_ATTENTION` | FlashAttention 커널. 원래 CUDA GPU를 위해 설계됨 | 주로 CUDA GPU (환경에 따라 CPU 지원이 추가되기도 함) |
| `EFFICIENT_ATTENTION` | memory-efficient attention이라는 또 다른 최적화 커널 | 주로 CUDA GPU |
| `CUDNN_ATTENTION` | NVIDIA cuDNN이 제공하는 attention 커널 | CUDA GPU (최신 환경) |

어떤 backend가 선택될지는 GPU 종류, PyTorch 버전, dtype, head 차원,
시퀀스 길이 등 여러 조건에 따라 달라지고, 버전이 올라가면서 계속 바뀌고 있습니다.
그러니 "이렇게 하면 무조건 FlashAttention이 선택된다"고 외우기보다는,
아래처럼 **직접 backend를 지정해서 지금 이 환경에서 무엇이 되는지 확인**하는
습관을 들이는 것이 훨씬 안전합니다.


In [ ]:
# ============================================================
# STEP 5-1. SDPA 기본 사용법 (자동 backend 선택)
# ============================================================
from torch.nn.functional import scaled_dot_product_attention

# 이번에는 실제 모델에 가까운 크기로 실험합니다.
# (batch_size, num_heads, seq_len, head_dim)
#   batch_size = 한 번에 처리하는 문장(시퀀스) 수
#   num_heads  = multi-head attention에서 head의 개수
#   seq_len    = 문장의 토큰 개수
#   head_dim   = 각 head가 다루는 벡터 차원 (보통 d_model / num_heads)
batch_size = 2
num_heads = 8
seq_len = 1024
head_dim = 64

# FlashAttention 커널은 float16 / bfloat16 같은 半정밀도(half precision)에서
# 동작하도록 설계되었습니다. (CPU에서는 지원 범위가 제한적이라 float32를 사용합니다)
dtype = torch.float16 if device.type == "cuda" else torch.float32

q = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device, dtype=dtype)
k = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device, dtype=dtype)
v = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device, dtype=dtype)

print("=== 입력 shape / dtype ===")
print(f"q shape: {tuple(q.shape)}, dtype: {q.dtype}")
print(f"k shape: {tuple(k.shape)}, dtype: {k.dtype}")
print(f"v shape: {tuple(v.shape)}, dtype: {v.dtype}")

# is_causal=True: 언어모델처럼 "미래 토큰을 보지 못하게" 하는 마스킹을
#                 함수 내부에서 자동으로 적용해줍니다.
#                 (STEP 2-3에서 우리가 직접 만든 causal_mask를 대신 처리해주는 옵션입니다)
output = scaled_dot_product_attention(q, k, v, is_causal=True)

print()
print("=== SDPA 실행 결과 ===")
print(f"output shape: {tuple(output.shape)}  # 입력 q와 동일한 shape")
print("-> PyTorch가 현재 환경에 맞는 backend를 자동으로 선택해 계산했습니다.")
print("   (어떤 backend가 선택됐는지는 다음 셀(STEP 5-2)에서 직접 확인해봅니다.)")


In [ ]:
# ============================================================
# STEP 5-2. 실제로 어떤 backend를 사용할 수 있는지 직접 확인하기
# ============================================================
# torch.nn.attention.sdpa_kernel 컨텍스트 매니저를 사용하면
# "이 backend만 사용해라"라고 강제할 수 있습니다.
# 해당 backend를 지금 이 입력/환경에서 사용할 수 없다면 에러가 발생하는데,
# 이를 이용해서 "정말 사용 가능한 backend가 무엇인지" 직접 확인할 수 있습니다.

from torch.nn.attention import SDPBackend, sdpa_kernel

def try_backend(name, backend, q, k, v):
    """지정한 backend로 SDPA를 실행해보고, 성공/실패 여부를 알려주는 helper 함수"""
    try:
        with sdpa_kernel(backend):
            out = scaled_dot_product_attention(q, k, v, is_causal=True)
        print(f"[성공] {name} backend 사용 가능 -> output shape {tuple(out.shape)}")
        return out
    except RuntimeError as e:
        print(f"[실패] {name} backend를 사용할 수 없는 환경입니다.")
        print(f"        (사유 요약: {str(e).splitlines()[0]})")
        return None

print(f"현재 device: {device}")
print("각 backend가 지원되는지 하나씩 확인합니다:\n")
math_output = try_backend("MATH (naive와 유사)", SDPBackend.MATH, q, k, v)
flash_output = try_backend("FLASH_ATTENTION", SDPBackend.FLASH_ATTENTION, q, k, v)
efficient_output = try_backend("EFFICIENT_ATTENTION", SDPBackend.EFFICIENT_ATTENTION, q, k, v)

# 뒤에서 계속 재사용할 수 있도록 "FlashAttention을 쓸 수 있는가"를 flag로 저장해둡니다.
flash_available = flash_output is not None

print()
if flash_available:
    print("-> 이 환경에서는 FLASH_ATTENTION backend를 사용할 수 있습니다.")
    print("   단, CPU에서 성공했다면 이는 PyTorch가 이식성을 위해 제공하는 구현일 수 있고,")
    print("   STEP 4에서 설명한 'GPU HBM/SRAM' 기반의 절약 효과와는 다를 수 있습니다.")
else:
    print("-> 이 환경에서는 FLASH_ATTENTION backend를 사용할 수 없습니다.")
    print("   STEP 6~9의 FlashAttention 관련 비교는 안내 메시지로 대체됩니다.")


## STEP 6. FlashAttention도 정말 "같은 답"을 내는지 검증하기

FlashAttention은 계산을 처리하는 순서(메모리를 다루는 방식)만 다를 뿐,
수학적으로는 STEP 2의 naive 구현과 **동일한 결과**를 내야 합니다.
직접 두 결과를 비교해서 확인해봅시다.


In [ ]:
# ============================================================
# STEP 6. 결과값 비교 (Naive vs FlashAttention)
# ============================================================
# 공정한 비교를 위해 같은 입력(q, k, v)에 대해 두 가지 방식으로 각각 계산합니다.
# float16은 정밀도가 낮아 미세한 오차가 생기기 쉬우므로,
# 비교를 위해 float32로 변환한 뒤 진행합니다.
q32, k32, v32 = q.float(), k.float(), v.float()

naive_result = naive_scaled_dot_product_attention(q32, k32, v32, is_causal=True)

if flash_available:
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        flash_result = scaled_dot_product_attention(q32, k32, v32, is_causal=True)

    max_diff = (naive_result - flash_result).abs().max().item()
    is_close = torch.allclose(naive_result, flash_result, atol=1e-4, rtol=1e-4)

    print(f"두 결과의 최대 차이(max abs diff): {max_diff:.8f}")
    print(f"torch.allclose(naive, flash) 결과: {is_close}")
    print()
    if is_close:
        print("[검증 성공] FlashAttention이 naive 구현과 (부동소수점 오차 범위 내에서)")
        print("            정확히 같은 결과를 낸다는 것을 확인했습니다!")
else:
    print("이 환경에서는 FLASH_ATTENTION backend를 사용할 수 없어 비교를 건너뜁니다.")
    print("(위 STEP 5-2 결과를 참고하세요. CUDA GPU 환경에서 다시 실행해보세요.)")


## STEP 7. 메모리 사용량 비교

STEP 3~4에서 예상했던 메모리 절약 효과를 실제로 측정해봅니다.
`torch.cuda.max_memory_allocated()`를 이용하면 특정 구간에서
GPU가 실제로 사용한 최대(peak) 메모리를 MB 단위로 측정할 수 있습니다.

측정 순서:
1. `torch.cuda.reset_peak_memory_stats()`로 카운터를 초기화합니다.
2. 측정하고 싶은 연산을 실행합니다.
3. `torch.cuda.max_memory_allocated()`로 그 구간의 최대 사용량을 읽습니다.

이 실험은 CUDA GPU 환경에서만 의미가 있으므로, CPU 환경에서는
안내 메시지만 출력하고 넘어갑니다.


In [ ]:
# ============================================================
# STEP 7. 메모리 사용량 비교 (Naive vs FlashAttention)
# ============================================================

def measure_peak_memory_mb(fn, *args, **kwargs):
    """
    함수 fn(*args, **kwargs)를 실행하는 동안 GPU가 사용한
    peak(최대) 메모리를 MB 단위로 반환하는 helper 함수.
    (CUDA 환경에서만 호출하세요.)
    """
    torch.cuda.synchronize()              # 이전 GPU 작업이 다 끝날 때까지 대기
    torch.cuda.empty_cache()              # PyTorch가 캐싱해둔 미사용 메모리 반환
    torch.cuda.reset_peak_memory_stats()  # peak 측정 카운터 초기화

    _ = fn(*args, **kwargs)               # 측정하려는 연산 실행

    torch.cuda.synchronize()              # 연산이 끝날 때까지 대기 (GPU는 비동기 실행!)
    peak_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    return peak_mb


# seq_len이 클수록 naive와 FlashAttention의 차이가 뚜렷해집니다.
# (CPU 환경에서는 STEP 8의 naive 속도 측정에만 재사용할 수 있도록
#  더 작은 크기로 텐서만 미리 만들어둡니다.)
seq_len_test = 4096 if device.type == "cuda" else 1024
q_big = torch.randn(batch_size, num_heads, seq_len_test, head_dim, device=device, dtype=dtype)
k_big = torch.randn(batch_size, num_heads, seq_len_test, head_dim, device=device, dtype=dtype)
v_big = torch.randn(batch_size, num_heads, seq_len_test, head_dim, device=device, dtype=dtype)

def run_flash(q, k, v, is_causal):
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        return scaled_dot_product_attention(q, k, v, is_causal=is_causal)

if device.type == "cuda":
    naive_mem = measure_peak_memory_mb(
        naive_scaled_dot_product_attention, q_big, k_big, v_big, is_causal=True
    )
    print(f"Naive 구현      peak memory : {naive_mem:>10.1f} MB  (seq_len={seq_len_test})")

    if flash_available:
        flash_mem = measure_peak_memory_mb(run_flash, q_big, k_big, v_big, is_causal=True)
        print(f"FlashAttention  peak memory : {flash_mem:>10.1f} MB  (seq_len={seq_len_test})")
        print(f"-> 메모리 절약 비율: {naive_mem / flash_mem:.1f}배 적게 사용")
    else:
        print("[안내] 이 환경에서는 FLASH_ATTENTION backend를 사용할 수 없습니다. (STEP 5-2 결과 참고)")
else:
    print("[안내] 메모리 비교 실험은 CUDA GPU 환경에서만 의미가 있어 생략합니다.")
    print(f"      (현재 device: {device})")
    print("      대신 STEP 3에서 계산한 이론적 예측치를 참고하세요:")
    print(f"      seq_len={seq_len_test}일 때 naive 방식이 만드는 attention 행렬 이론값 = "
          f"{attention_matrix_memory_mb(batch_size, num_heads, seq_len_test):.1f} MB")
    print("      (이것은 attention 행렬 '하나'의 크기이며, 실제 peak 메모리는 이보다 더 큽니다.)")


## STEP 8. 속도 비교

이번에는 실행 시간을 비교합니다. 한 가지 중요한 주의사항이 있습니다.
**GPU 연산은 비동기(asynchronous)로 실행됩니다.** CPU 코드는 GPU에게
"이 연산을 해줘"라고 요청만 하고 바로 다음 줄로 넘어가 버리기 때문에,
`torch.cuda.synchronize()`로 "GPU 작업이 진짜로 다 끝날 때까지 기다려라"라고
명시하지 않으면 시간을 잘못 측정하게 됩니다.


In [ ]:
# ============================================================
# STEP 8. 속도 비교 (Naive vs FlashAttention)
# ============================================================
def measure_time_ms(fn, *args, n_repeat=10, **kwargs):
    """n_repeat번 반복 실행한 뒤 평균 실행 시간(ms)을 반환하는 helper 함수"""
    # warm-up: 첫 실행은 GPU 초기화/캐시 준비 등으로 느릴 수 있어 측정에서 제외합니다.
    _ = fn(*args, **kwargs)
    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.time()
    for _ in range(n_repeat):
        _ = fn(*args, **kwargs)
    if device.type == "cuda":
        torch.cuda.synchronize()
    end = time.time()

    return (end - start) / n_repeat * 1000  # ms 단위로 변환


naive_time = measure_time_ms(
    naive_scaled_dot_product_attention, q_big, k_big, v_big, is_causal=True
)
print(f"Naive 구현      평균 실행 시간: {naive_time:>10.3f} ms  (seq_len={seq_len_test})")

if device.type == "cuda" and flash_available:
    flash_time = measure_time_ms(run_flash, q_big, k_big, v_big, is_causal=True)
    print(f"FlashAttention  평균 실행 시간: {flash_time:>10.3f} ms  (seq_len={seq_len_test})")
    print(f"-> 속도 향상: {naive_time / flash_time:.1f}배 빠름")
else:
    print("[안내] FlashAttention과의 속도 비교는 CUDA GPU 환경에서만 진행합니다.")


## STEP 9. 직접 실험해보기 (Experiment Zone)

아래 코드는 `seq_len`을 여러 값으로 바꿔가며, naive 구현과
FlashAttention의 메모리 차이가 시퀀스 길이가 길어질수록
어떻게 벌어지는지 관찰합니다. (CUDA GPU 환경 전용 실험입니다.)

**직접 바꿔보면 좋은 것들:**
- `seq_len_list`의 값을 더 크게(예: 16384) 바꿔보면 어떻게 될까요?
  (naive 구현은 메모리 부족(OOM) 에러가 날 수도 있습니다 — 그 자체가 중요한 관찰 포인트입니다!)
- `num_heads`나 `head_dim`을 바꾸면 결과가 어떻게 달라질까요?
- `dtype`을 `torch.bfloat16`으로 바꾸면 어떻게 될까요?


In [ ]:
# ============================================================
# STEP 9. seq_len에 따른 메모리 사용량 변화 실험
# ============================================================
if device.type == "cuda" and flash_available:
    seq_len_list = [512, 1024, 2048, 4096, 8192]

    print(f"{'seq_len':>8} | {'Naive (MB)':>14} | {'Flash (MB)':>12} | {'절약 비율':>10}")
    print("-" * 54)

    for sl in seq_len_list:
        q_e = torch.randn(1, num_heads, sl, head_dim, device=device, dtype=dtype)
        k_e = torch.randn(1, num_heads, sl, head_dim, device=device, dtype=dtype)
        v_e = torch.randn(1, num_heads, sl, head_dim, device=device, dtype=dtype)

        naive_mem_e = None
        try:
            naive_mem_e = measure_peak_memory_mb(
                naive_scaled_dot_product_attention, q_e, k_e, v_e, is_causal=True
            )
            naive_str = f"{naive_mem_e:.1f}"
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                naive_str = "OOM (메모리 부족)"
                torch.cuda.empty_cache()
            else:
                raise

        flash_mem_e = measure_peak_memory_mb(run_flash, q_e, k_e, v_e, is_causal=True)
        ratio_str = f"{naive_mem_e / flash_mem_e:.1f}x" if naive_mem_e else "-"

        print(f"{sl:>8} | {naive_str:>14} | {flash_mem_e:>12.1f} | {ratio_str:>10}")

    print()
    print("-> seq_len이 커질수록 naive 구현의 메모리 사용량이 급격히 늘어나")
    print("   FlashAttention과의 격차가 점점 벌어지는 것을 확인할 수 있습니다.")
    print("   (naive 구현은 O(N^2), FlashAttention은 O(N)에 가깝게 메모리를 사용합니다.)")
else:
    print(f"[안내] 이 실험은 CUDA GPU 환경 전용입니다. (현재 device: {device})")
    print("      Google Colab에서 [런타임 > 런타임 유형 변경 > GPU]를 선택한 뒤")
    print("      다시 실행해보세요.")


## STEP 10. 핵심만 다시 보기 (Cheat Sheet)

지금까지 원리를 자세히 살펴봤습니다. 실전에서 FlashAttention을 사용할 때
필요한 코드는 사실 아주 짧습니다. 아래는 지금까지 배운 내용을 바탕으로
주석을 단 "핵심 사용 패턴"입니다. 나중에 다시 참고할 때는 이 셀만 봐도
충분하도록 정리했습니다.


In [ ]:
# ============================================================
# STEP 10. 핵심 사용 패턴 요약 (Cheat Sheet)
# ============================================================
import torch
from torch.nn.functional import scaled_dot_product_attention

# 1) Q, K, V 준비
#    shape: (batch_size, num_heads, seq_len, head_dim)
#    dtype: float16 또는 bfloat16 권장 (FlashAttention 커널이 선호하는 정밀도)
q_final = torch.randn(2, 8, 1024, 64, device=device, dtype=dtype)
k_final = torch.randn(2, 8, 1024, 64, device=device, dtype=dtype)
v_final = torch.randn(2, 8, 1024, 64, device=device, dtype=dtype)

# 2) 함수 호출 한 줄이면 끝
#    is_causal=True  -> 언어모델(GPT 등)처럼 미래 토큰을 막는 마스킹을 자동 적용
#    어떤 backend가 쓰일지는 PyTorch가 환경에 맞게 자동으로 선택 (STEP 5 참고)
output_final = scaled_dot_product_attention(q_final, k_final, v_final, is_causal=True)
print(f"Output shape: {tuple(output_final.shape)}")

# 3) (선택) 메모리 확인 - CUDA GPU 환경에서만 의미가 있음
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
    _ = scaled_dot_product_attention(q_final, k_final, v_final, is_causal=True)
    print(f"Peak memory: {torch.cuda.max_memory_allocated() / 1e6:.1f} MB")
else:
    print("[안내] 메모리 측정은 CUDA GPU 환경에서 실행해보세요.")


## STEP 11. 정리 (Summary) 및 참고문헌

### 정리

이번 노트북에서 확인한 내용을 정리하면:

1. **Attention의 기본 계산**: Q, K, V로부터 (seq_len × seq_len) 크기의
   attention score 행렬을 만들고, softmax와 V를 곱해 최종 출력을 얻습니다.
2. **Causal mask**: 미래 토큰 위치를 -inf로 채운 뒤 softmax를 적용하면
   해당 위치의 확률이 정확히 0이 되어, "미래를 보지 못하게" 만들 수 있습니다.
3. **문제**: attention score 행렬은 시퀀스 길이의 제곱(N²)에 비례해 커지고,
   이를 GPU의 느린 메모리(HBM)에 여러 번 쓰고 읽는 과정이 병목이 됩니다.
4. **FlashAttention의 해법**: 계산을 작은 블록(tile) 단위로 나누어
   빠른 메모리(SRAM) 안에서 한 번에(fused) 처리함으로써, **수학적으로는
   완전히 동일한 결과**를 내면서도 메모리 사용량과 속도를 모두 개선합니다.
5. **PyTorch 사용법**: `torch.nn.functional.scaled_dot_product_attention`
   하나로 여러 backend(MATH / FLASH_ATTENTION / EFFICIENT_ATTENTION /
   CUDNN_ATTENTION)를 자동 또는 명시적으로 선택해 사용할 수 있습니다.

### 다음 노트북에서는?

이 노트북은 "사용법"과 "왜 필요한가"에 집중했습니다. 다음 실습 노트북에서는
FlashAttention 내부에서 tiling과 online softmax가 구체적으로 어떤 순서로
값을 갱신하는지, 간단한 Python 코드로 직접 흉내내어 구현해보며 원리를
더 깊이 이해해보겠습니다.

### 참고문헌

- Tri Dao, Daniel Y. Fu, Stefano Ermon, Atri Rudra, Christopher Ré,
  "FlashAttention: Fast and Memory-Efficient Exact Attention with
  IO-Awareness", NeurIPS 2022.
- Tri Dao, "FlashAttention-2: Faster Attention with Better Parallelism
  and Work Partitioning", 2023 (ICLR 2024).
- PyTorch 공식 문서: `torch.nn.functional.scaled_dot_product_attention`,
  `torch.nn.attention.sdpa_kernel`
  (https://docs.pytorch.org 에서 최신 문서를 확인하세요.)
